In [16]:
# ①ライブラリのインポート
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, roc_auc_score

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [17]:
# ② データ読み込み（正しいパス）
p = "/kaggle/input/competitions/playground-series-s6e4/"

df_train = pd.read_csv(p + "train.csv")
df_test = pd.read_csv(p + "test.csv")
df_submission = pd.read_csv(p + "sample_submission.csv")

df_train.head()


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [18]:
# ③ データの基本確認
df_train.info()
df_train.describe()
df_train.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

In [19]:
# ④ 目的変数と特徴量の分離
target = "Irrigation_Need"  # ← 必ず train.csv を見て正しい名前に変更

X = df_train.drop(columns=[target])
y = df_train[target]


In [20]:
# ⑤.5 カテゴリ変数を Label Encoding（追加セル）

from sklearn.preprocessing import LabelEncoder

# X の中で dtype が object（文字列）の列を探す
cat_cols = X.select_dtypes(include=["object"]).columns

# 各カテゴリ列を Label Encoding
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    df_test[col] = le.transform(df_test[col])
    le_dict[col] = le  # 必要なら保存


In [21]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = []
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold+1}")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)

    score = accuracy_score(y_valid, pred_valid)
    fold_scores.append(score)

    print(f"  Accuracy: {score:.5f}")

print("\nCV Mean:", np.mean(fold_scores))


Fold 1
  Accuracy: 0.98545
Fold 2
  Accuracy: 0.98502
Fold 3
  Accuracy: 0.98484
Fold 4
  Accuracy: 0.98533
Fold 5
  Accuracy: 0.98521

CV Mean: 0.9851698412698411


In [22]:
# ⑥ モデル定義

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)


In [23]:
# ⑦ 学習
model.fit(X_train, y_train)


RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [24]:
# ⑧ 検証スコアを確認

pred_valid = model.predict(X_valid)
accuracy_score(y_valid, pred_valid)


0.9852063492063492

In [25]:
# ⑨ テストデータで予測

pred_test = model.predict(df_test)


In [26]:

# ⑩ 提出ファイル作成
df_submission["Irrigation_Need"] = pred_test
df_submission.to_csv("submission.csv", index=False)
df_submission.head()


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
